In [22]:
import pandas as pd
import os
import pinecone
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

load_dotenv()

True

In [5]:
files = pd.read_csv("course_descriptions.csv", encoding="cp1252")

In [6]:
files.head()

,course_name,course_slug,course_technology,course_description,course_topic,course_description_short
0,Introduction to Tableau,tableau,tableau,Tableau is now one of the most popular busines...,data visualization,Teaching you how to tell compelling stories wi...
1,The Complete Data Visualization Course with Py...,data-visualization,python,The Data Visualization course is designed for ...,data visualization,Teaching you how to master the art of creating...
2,Introduction to R Programming,introduction-to-r-programming,r,R is one of the best programming languages spe...,programming,"Providing you with the skills to manipulate, a..."
3,Data Preprocessing with NumPy,data-preprocessing-numpy,python,This course is designed to show you how to wor...,data processing,This course will guide you through one of Pyth...
4,Introduction to Data and Data Science,intro-to-data-and-data-science,theory,Working with data is an essential part of main...,machine learning,Introducing you to the field of data science a...


In [9]:
def create_course_description(row):
    return f"""The course name is {row["course_name"]}, the slug is {row["course_slug"]}, the technologu is {row["course_technology"]}, and the topic is {row["course_topic"]}."""

In [10]:
files["course_description_new"] = files.apply(create_course_description, axis=1)
files.head()

,course_name,course_slug,course_technology,course_description,course_topic,course_description_short,course_description_new
0,Introduction to Tableau,tableau,tableau,Tableau is now one of the most popular busines...,data visualization,Teaching you how to tell compelling stories wi...,"The course name is Introduction to Tableau, th..."
1,The Complete Data Visualization Course with Py...,data-visualization,python,The Data Visualization course is designed for ...,data visualization,Teaching you how to master the art of creating...,The course name is The Complete Data Visualiza...
2,Introduction to R Programming,introduction-to-r-programming,r,R is one of the best programming languages spe...,programming,"Providing you with the skills to manipulate, a...",The course name is Introduction to R Programmi...
3,Data Preprocessing with NumPy,data-preprocessing-numpy,python,This course is designed to show you how to wor...,data processing,This course will guide you through one of Pyth...,The course name is Data Preprocessing with Num...
4,Introduction to Data and Data Science,intro-to-data-and-data-science,theory,Working with data is an essential part of main...,machine learning,Introducing you to the field of data science a...,The course name is Introduction to Data and Da...


In [15]:
pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY"), environment=os.getenv("PINECONE_ENVIRONMENT")
)

In [17]:
index_name = "my-index"
dimensions = 384
metric = "cosine"

In [18]:
if index_name in [index.name for index in pc.list_indexes()]:
    print(f"Index '{index_name}' already exists.")
    pc.delete_index(index_name)
    print(f"Index '{index_name}' deleted.")
else:
    print(f"Index '{index_name}' does not exist.")

Index 'my-index' does not exist.


In [20]:
pc.create_index(
    name=index_name,
    dimension=dimensions,
    metric=metric,
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

{
    "name": "my-index",
    "metric": "cosine",
    "host": "my-index-h2xw5gc.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "access-control-allow-origin": "*",
            "vary": "origin,access-control-request-method,access-control-request-headers",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2025-10",
  

In [21]:
index = pc.Index(index_name)

/opt/anaconda3/envs/langchain_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Embedding the data


In [23]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [24]:
def create_embeddings(row):
    combined_text = " ".join(
        [
            str(row[field])
            for field in [
                "course_description",
                "course_description_new",
                "course_description_short",
            ]
        ]
    )
    embedding = model.encode(combined_text, show_progress_bar=False).tolist()
    return embedding

In [25]:
files["embeddings"] = files.apply(create_embeddings, axis=1)

In [27]:
vectors_to_upsert = [
    (str(row["course_name"]), row["embeddings"]) for _, row in files.iterrows()
]

index.upsert(vectors=vectors_to_upsert)

print("Vectors upserted successfully.")

Vectors upserted successfully.


In [28]:
files.head()

,course_name,course_slug,course_technology,course_description,course_topic,course_description_short,course_description_new,embeddings
0,Introduction to Tableau,tableau,tableau,Tableau is now one of the most popular busines...,data visualization,Teaching you how to tell compelling stories wi...,"The course name is Introduction to Tableau, th...","[0.04754232242703438, -0.025731457397341728, -..."
1,The Complete Data Visualization Course with Py...,data-visualization,python,The Data Visualization course is designed for ...,data visualization,Teaching you how to master the art of creating...,The course name is The Complete Data Visualiza...,"[0.045970506966114044, -0.0190302524715662, -0..."
2,Introduction to R Programming,introduction-to-r-programming,r,R is one of the best programming languages spe...,programming,"Providing you with the skills to manipulate, a...",The course name is Introduction to R Programmi...,"[-0.032060395926237106, 0.003249592613428831, ..."
3,Data Preprocessing with NumPy,data-preprocessing-numpy,python,This course is designed to show you how to wor...,data processing,This course will guide you through one of Pyth...,The course name is Data Preprocessing with Num...,"[-0.042630571871995926, -0.011502289213240147,..."
4,Introduction to Data and Data Science,intro-to-data-and-data-science,theory,Working with data is an essential part of main...,machine learning,Introducing you to the field of data science a...,The course name is Introduction to Data and Da...,"[0.003696816274896264, 0.013247908093035221, -..."


### Semantic search


In [30]:
query = "clustering"
query_embedding = model.encode(query).tolist()

In [31]:
query_results = index.query(
    vector=[query_embedding], top_k=12, include_metadata=True, include_values=True
)

In [32]:
query_results

QueryResponse(matches=[{'id': 'Machine Learning in Excel',
 'score': 0.358200073,
 'values': [0.0033723868,
            -0.0292600784,
            -0.0221662056,
            -0.0199606232,
            -0.0260353759,
            -0.0285246894,
            -0.0483845398,
            -0.0468946844,
            0.00338905421,
            0.0334535874,
            -0.0416799746,
            -0.0374811329,
            0.0728667751,
            -0.0349725336,
            0.00254065124,
            0.0390005931,
            -0.00747278472,
            -0.00826809,
            0.000724587124,
            -0.0621499643,
            0.0899556577,
            0.0292359237,
            -0.046205081,
            -0.0267778076,
            0.0370306559,
            0.0199046358,
            0.0608185865,
            -0.0015481452,
            -0.0428084768,
            -0.0307227261,
            -0.0565921925,
            0.0206759945,
            0.00639701681,
            0.0444554053,
            

In [33]:
for match in query_results.matches:
    print(f"Course Name: {match.id}, Score: {match.score}")

Course Name: Machine Learning in Excel, Score: 0.358200073
Course Name: Machine Learning with K-Nearest Neighbors, Score: 0.340294361
Course Name: Customer Churn Analysis with SQL and Tableau, Score: 0.297932625
Course Name: Linear Algebra and Feature Selection, Score: 0.271223098
Course Name: Machine Learning in Python, Score: 0.265728
Course Name: Growth Analysis with SQL, Python, and Tableau  , Score: 0.260608673
Course Name: Fashion Analytics with Tableau, Score: 0.25161314
Course Name: Customer Engagement Analysis with SQL and Tableau, Score: 0.240436569
Course Name: Machine Learning with Naive Bayes, Score: 0.237494484
Course Name: Data Analysis with Excel Pivot Tables, Score: 0.233613014
Course Name: Data Preprocessing with NumPy, Score: 0.227066055
Course Name: Machine Learning with Support Vector Machines, Score: 0.226127625
